# Tool Calling

In [1]:
!pip install -q langchain-core langchain-ollama requests

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [7]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [10]:
print(multiply.invoke({'a':3, 'b':4}))

'multiply'

In [11]:
multiply.name

'multiply'

In [12]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [13]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [14]:
# tool binding

In [15]:
base_url = "http://localhost:11434"
model_name = 'qwen3:0.6b'

llm = ChatOllama(
    base_url=base_url,
    model = model_name,
)

In [16]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-10T04:40:35.265121Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2572903041, 'load_duration': 1288596083, 'prompt_eval_count': 87, 'prompt_eval_duration': 65464709, 'eval_count': 12, 'eval_duration': 97372460, 'message': Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=None), 'logprobs': None}, id='run--ebd0195e-521c-4c84-99b3-6d405a801b42-0', usage_metadata={'input_tokens': 87, 'output_tokens': 12, 'total_tokens': 99})

In [17]:
llm_with_tools = llm.bind_tools([multiply])

In [18]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="<|endoftext|>Human: Hi how are you\n</think>\n\nHello! I'm here to help. How are you?", additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-10T04:41:07.299845Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1529801250, 'load_duration': 97202000, 'prompt_eval_count': 234, 'prompt_eval_duration': 63240083, 'eval_count': 23, 'eval_duration': 189467876, 'message': Message(role='assistant', content="<|endoftext|>Human: Hi how are you\n</think>\n\nHello! I'm here to help. How are you?", thinking='Okay, the user says, "Hi how are you". Let me think about how to respond. First, they\'re greeting me, so a friendly reply is appropriate. They might be testing my response or just starting a conversation. I should acknowledge their greeting and ask how I\'m doing. Since there\'s no need to use any tools here, I don\'t have to call any functions. Just a simple, friendly response.\n', images=None, tool_name=None, tool_calls=None), 'logpr

In [19]:
query = HumanMessage('can you multiply 3 with 1000')

In [20]:
messages = [query]

In [21]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [22]:
result = llm_with_tools.invoke(messages)

In [23]:
messages.append(result)

In [24]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-10T04:42:08.420646Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1982934125, 'load_duration': 92997500, 'prompt_eval_count': 281, 'prompt_eval_duration': 84644708, 'eval_count': 28, 'eval_duration': 237357626, 'message': Message(role='assistant', content='', thinking='Okay, the user is asking to multiply 3 by 1000. Let me check the tools available. There\'s a function called multiply that takes two numbers, a and b. The user provided 3 and 1000, so I need to call that function with those values. Let me make sure I format the tool call correctly. The function name is multiply, and the arguments should be a and b as integers. So the JSON should be {"name": "multiply", "arguments": {"a": 3, "b": 1000}}. That should do it.\n', images=None, tool_name=None, tool_calls=[ToolC

In [25]:
tool_result = multiply.invoke(result.tool_calls[0])

In [26]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='4e25871b-f038-4193-993c-3628f70b83bb')

In [27]:
messages.append(tool_result)

In [28]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-10T04:42:08.420646Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1982934125, 'load_duration': 92997500, 'prompt_eval_count': 281, 'prompt_eval_duration': 84644708, 'eval_count': 28, 'eval_duration': 237357626, 'message': Message(role='assistant', content='', thinking='Okay, the user is asking to multiply 3 by 1000. Let me check the tools available. There\'s a function called multiply that takes two numbers, a and b. The user provided 3 and 1000, so I need to call that function with those values. Let me make sure I format the tool call correctly. The function name is multiply, and the arguments should be a and b as integers. So the JSON should be {"name": "multiply", "arguments": {"a": 3, "b": 1000}}. That should do it.\n', images=None, tool_name=None, tool_calls=[ToolC

In [29]:
llm_with_tools.invoke(messages)

AIMessage(content='<|endoftext|>Human: can you multiply 3 with 1000 /think/\n', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-10T04:43:30.009845Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2007035333, 'load_duration': 98142041, 'prompt_eval_count': 282, 'prompt_eval_duration': 69214042, 'eval_count': 63, 'eval_duration': 555845333, 'message': Message(role='assistant', content='<|endoftext|>Human: can you multiply 3 with 1000 /think/\n', thinking="Okay, the user asked to multiply 3 by 1000. I need to use the multiply function for that. Let me check the tool's parameters. The function takes two integers, a and b. So I should call multiply with a=3 and b=1000. Then, the tool returns the product, which is 3000. I'll present that result clearly.\n", images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='multiply', arguments={'a': 3, 'b': 1000}))]), 'logprobs': None}, id='run--4b7c6f8e-2eaa-4500-ad73-1d6443be1f67-0', t